# 进阶教程（二）：LangGraph 状态图设计模式

> 设计模式 = 可复用的图结构解决方案。本讲沉淀 4 个高频模式 + 反模式清单。

## 本讲内容
| 模式 | 一句话 | 典型场景 |
|---|---|---|
| 子图封装 | 把循环图打包成一个节点 | 复用、分层设计 |
| Command 路由 | 节点内部决定下一步 | 动态审批流 |
| Send 并行归约 | 运行时动态分发 Map-Reduce | 多角度评审、批处理 |
| 校验-重试 | 哨兵节点把关输出 | 结构化输出兜底 |

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. 模式一：子图封装（Subgraph as Node）

"生成 → 校验 → 不合格回炉"的循环图，打包后挂进父图当普通节点。
父图代码完全不感知内部循环——**图的节点可以是图**：

In [3]:

from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class InnerState(TypedDict):
    text: str
    attempts: int

def write(state: InnerState):
    tip = "" if state["attempts"] == 0 else f"（注意修正：{state['tip'] if 'tip' in state else ''}）"
    r = model.invoke(f"写一句不超过 20 字的产品宣传语，主题：{state['text'][:20]}{tip}")
    return {"text": r.content}

def check(state: InnerState):
    ok = len(state["text"]) <= 25
    return {"text": state["text"], "attempts": state["attempts"] + 1,
            **({} if ok else {"tip": "太长了"})}

def route(state: InnerState):
    return "write" if (len(state["text"]) > 25 and state["attempts"] < 3) else "pass"

inner = StateGraph(InnerState)
inner.add_node("write", write)
inner.add_node("check", check)
inner.add_edge(START, "write")
inner.add_edge("write", "check")
inner.add_conditional_edges("check", route, {"write": "write", "pass": END})
slogan_factory = inner.compile()   # 子图编译产物 = Runnable

# ---- 子图作为父图节点 ----
class OuterState(TypedDict):
    product: str
    slogan: str

def slogan_node(state: OuterState):
    r = slogan_factory.invoke({"text": state["product"], "attempts": 0})
    return {"slogan": r["text"]}

outer = StateGraph(OuterState)
outer.add_node("make_slogan", slogan_node)
outer.add_edge(START, "make_slogan")
outer.add_edge("make_slogan", END)
app = outer.compile()
print(app.invoke({"product": "保温杯", "slogan": ""})["slogan"])

好的，这里有几个不同风格的宣传语，供您选择：

1. **极简风**：好，是刚刚好。
2. **品质风**：好，经得起时间考验。
3. **情感风**：好，让生活更从容。
4. **自信风**：好，无需多言。

（注：已修正原句“供您选”为“供您选择”，并补充了完整宣传语。）


**要点**：
- 子图与父图**状态独立**（InnerState / OuterState），靠节点函数做输入输出适配
- `create_agent(...)` 返回的也是 CompiledStateGraph，同样可以直接 `add_node` 挂载

## 2. 模式二：Command 路由（节点内决策）

传统 `add_conditional_edges` 把路由逻辑放在节点外；
`Command` 让节点**在返回结果的同时宣布下一步去哪**——
决策与执行内聚，特别适合"执行完才知道下一步"的场景：

In [5]:

from langgraph.types import Command
from pydantic import BaseModel, Field

class ReviewResult(BaseModel):
    score: float = Field(description="质量分 0-10")
    comment: str = Field(description="一句话评语")

class FlowState(TypedDict):
    draft: str
    score: float
    history: Annotated[list, lambda old, new: old + new]

def draft_node(state: FlowState) -> Command:
    """写稿节点：根据上一轮评审分数决定重写还是交付"""
    if state.get("score", 0) >= 8:
        return Command(goto=END, update={"draft": state["draft"]})
    # 首次或低分回炉：写一版新文案（模型每次生成不同版本）
    r = model.invoke("写一句不超过 20 字的保温杯文案，朗朗上口、突出保温卖点。")
    return Command(goto="review", update={"draft": r.content})

def review_node(state: FlowState):
    r = model.with_structured_output(ReviewResult).invoke(
        f"给这句文案打分（0-10）并一句话点评：{state['draft']}")
    print(f"{state['draft']} ,[review] 分数 {r.score}: {r.comment}")
    return {"score": r.score}

b = StateGraph(FlowState)
b.add_node("draft", draft_node)      # 节点内部用 Command 宣布去向
b.add_node("review", review_node)
b.add_edge(START, "draft")
b.add_edge("review", "draft")        # 回边固定，是否再走由 draft 自己决定
flow = b.compile()
result = flow.invoke({"draft": "", "score": 0, "history": []},
                     config={"recursion_limit": 10})   # 循环安全阀
print("最终文案:", result["draft"])

**暖在口，热在心，一杯恒温。** ,[review] 分数 7.0: 对仗工整、朗朗上口，将温度与情感巧妙结合，但“恒温”一词略显直白，缺少一点诗意升华。
**“一口暖到心，热饮不降温。”** ,[review] 分数 7.0: 简洁押韵，情感温度到位，但“不降温”略显直白，缺少一点画面感和记忆点。
**“暖在心头，久伴不凉。”** ,[review] 分数 8.0: 简洁有力，意象温暖，对仗工整，适合保温杯等产品文案，但稍显常见，缺乏独特记忆点。
最终文案: **“暖在心头，久伴不凉。”**


**要点**：`Command(goto=..., update=...)` 中 update 写入状态、goto 决定路由；
回边 `review -> draft` 静态存在，但 draft 内部可直接 `goto=END` 跳出，
循环控制权收进节点内部，图结构保持极简。

## 3. 模式三：Send 动态并行（Map-Reduce）

`Send` 在**运行时**根据状态动态决定分发几份、每份带什么参数——
并行度不再写死在图结构里：

In [6]:

import operator
from langgraph.types import Send

class MapState(TypedDict):
    topic: str
    angles: list            # 评审角度（运行时才知道有几个）
    reviews: Annotated[list, operator.add]   # reducer：并行结果自动归并

class WorkerState(TypedDict):
    """Send 的 payload：只带本 worker 需要的字段"""
    topic: str
    angle: str

def dispatch(state: MapState):
    """Map：为每个角度发一个 worker（动态并行）"""
    return [Send("worker", {"topic": state["topic"], "angle": a})
            for a in state["angles"]]

def worker(ws: WorkerState):
    r = model.invoke(f"从「{ws['angle']}」角度，用一句话评价：{ws['topic']}")
    print(f"  [{ws['angle']}] 完成")
    return {"reviews": [f"[{ws['angle']}] {r.content}"]}

b = StateGraph(MapState)
b.add_node("worker", worker)
b.add_conditional_edges(START, dispatch, ["worker"])  # 入口直接动态分发
b.add_edge("worker", END)
mapreduce = b.compile()

r = mapreduce.invoke({"topic": "RAG 技术", "angles": ["实用性", "成本", "风险"], "reviews": []})
for line in r["reviews"]:
    print(line)

  [成本] 完成
  [风险] 完成
  [实用性] 完成
[实用性] RAG 技术是“用检索到的外部知识，给大模型装上实时更新的外挂记忆”，它最大的实用性在于——**用最低的成本（无需重新训练）解决了大模型“知识过时、胡编乱造、缺乏私有数据”三大痛点，但前提是你得接受它“检索不准就全盘皆输”的脆弱性。**
[成本] RAG 技术通过“外挂知识库”替代“全量参数重训”，本质上是**用可控的检索与生成算力成本，换取模型能力扩展的边际成本大幅下降**——但前提是，你得先接受它那套“隐形的”工程维护、存储和延迟成本。
[风险] RAG技术通过引入外部知识检索来约束生成，本质上是用“可控的信息来源”对冲“模型幻觉”的风险，但同时也将风险转移到了检索质量、知识时效性与上下文注入的可靠性上。


**要点**：
- worker 收到的是 **WorkerState**（Send payload），不是完整 MapState
- 多个 worker **并行**执行，各自的返回值经 `operator.add` reducer 自动归并
- 与 fan-out（`add_edge(["a","b"], "join")`）的区别：Send 的分支数量运行时可变

## 4. 模式四：校验-重试循环（哨兵节点）

LLM 输出不可信？在出口放一个**哨兵节点**做结构化校验，
不合法就带着错误信息回炉，超过次数走兜底——
rag_qa_project 的 grade_documents 正是该模式的检索版：

In [7]:

import re
import json as _json
from pydantic import BaseModel, Field, ValidationError

class Report(BaseModel):
    title: str = Field(min_length=2, max_length=20)
    risk_level: int = Field(ge=1, le=5)

class VS(TypedDict):
    demand: str
    raw: str
    report: dict
    attempts: int

def gen(state: VS):
    hint = "" if state["attempts"] == 0 else "上次输出不合法，请严格遵守 schema。"
    schema = _json.dumps({"title": "2-20字标题", "risk_level": "1到5的整数"},
                         ensure_ascii=False)
    r = model.invoke(
        f"严格按此 JSON 格式输出项目风险报告 {schema}。"
        f"需求：{state['demand']}。{hint}")
    return {"raw": r.content}

def validate(state: VS):
    """哨兵节点：只做校验，不做生成"""
    m = re.search(r"\{.*\}", state["raw"], re.S)
    try:
        obj = _json.loads(m.group()) if m else {}
        Report(**obj)          # Pydantic 校验失败会抛 ValidationError
        return {"report": obj, "attempts": state["attempts"] + 1}
    except ValidationError as e:
        return {"attempts": state["attempts"] + 1,
                "raw": f"ERROR: {str(e)[:200]}"}

def route(state: VS):
    if "ERROR" not in state["raw"] or state["attempts"] >= 3:
        return "done"
    return "retry"

b = StateGraph(VS)
b.add_node("gen", gen)
b.add_node("validate", validate)
b.add_edge(START, "gen")
b.add_edge("gen", "validate")
b.add_conditional_edges("validate", route, {"retry": "gen", "done": END})
validator = b.compile()
r = validator.invoke({"demand": "数据库迁移到云上", "raw": "", "report": {}, "attempts": 0})
print("最终报告:", r.get("report") or r["raw"][:150], "| 尝试次数:", r["attempts"])

最终报告: {'title': '数据库云迁移风险评估', 'risk_level': 3} | 尝试次数: 1


## 5. 反模式清单

| 反模式 | 症状 | 正解 |
|---|---|---|
| 巨型节点 | 一个函数又检索又生成又路由 | 单一职责，节点间用状态协作 |
| 隐式全局状态 | 节点读写外部变量 | 一切经过 State（可检查、可回放） |
| 无安全阀循环 | rewrite↔retrieve 死循环烧额度 | 计数器 + recursion_limit 双保险 |
| 状态字段未声明 | TypedDict 没写的字段被静默丢弃 | 补充 schema 声明（常见 KeyError 根因） |
| 并行写同一字段 | InvalidUpdateError | 各分支写独立字段，join 节点汇总 |

## 6. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| `Send` 的 payload 字段在目标节点读不到 | worker 状态与父状态不同构 | 为 worker 定义独立 TypedDict |
| Command 路由后图死循环 | 忘记 goto=END 的退出条件 | 循环体内必须有可达的出口 |
| 子图状态没传回父图 | 子图 update 的 key 不在父图 schema | 节点函数里做显式字段映射 |
| 归并结果乱序 | 并行节点写同一 list | 用 reducer（operator.add），接受非确定顺序或加序号 |